In [1]:
# Cell 1: Install required libraries
!pip install transformers sentence_transformers chromadb langchain huggingface_hub

In [2]:
# Cell 2: Import libraries and print API key
import os
import chromadb
from langchain.text_splitter import RecursiveCharacterTextSplitter
from sentence_transformers import SentenceTransformer
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch


/home/nikhil/anaconda3/lib/python3.11/site-packages/sentence_transformers/cross_encoder/CrossEncoder.py:11: TqdmExperimentalWarning: Using `tqdm.autonotebook.tqdm` in notebook mode. Use `tqdm.tqdm` instead to force console mode (e.g. in jupyter console)
  from tqdm.autonotebook import tqdm, trange


In [3]:
# Cell 3: Initialize text splitter and load files
# Initialize text splitter
text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=100)

# Load and split files
chunks = []
file_paths = [os.path.join("class_files", file) for file in os.listdir("class_files") if file.endswith('.rst')]

for file_path in file_paths:
    with open(file_path, 'r', encoding='utf-8') as file:
        content = file.read()
        chunks.extend(text_splitter.split_text(content))

print(f"Total chunks created: {len(chunks)}")

Total chunks created: 213


In [4]:
# Cell 4: Initialize embedding model and create collection
from sentence_transformers import SentenceTransformer

# Initialize the embedding model
embedding_model = SentenceTransformer('multi-qa-mpnet-base-dot-v1')

# Initialize Chroma client and create collection
client = chromadb.Client()
collection = client.create_collection(name="docs")

# Store each document in a vector embedding database
for i, d in enumerate(chunks):
    embedding = embedding_model.encode(d).tolist()
    collection.add(
        ids=[str(i)],
        embeddings=[embedding],
        documents=[d]
    )

modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/212 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/8.71k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

/home/nikhil/anaconda3/lib/python3.11/site-packages/huggingface_hub/file_download.py:1132: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


...

tokenizer_config.json:   0%|          | 0.00/363 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

1_Pooling/config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [5]:
# Cell 5: Generate embedding for prompt and retrieve relevant docs
# An example prompt
prompt = "Can you give me code to turn the servo to 45 degrees?"

# Generate an embedding for the prompt and retrieve the most relevant docs
prompt_embedding = embedding_model.encode(prompt).tolist()
results = collection.query(
    query_embeddings=[prompt_embedding],
    n_results=5
)

# Store the data
data = []

# Iterate through the results and store the documents
for doc_idx in range(len(results['ids'])):
    data.append(results['documents'][0][doc_idx].strip())

# Store the data as a large string
data_str = "".join(data)

In [11]:
# Cell 6: Initialize the LLM and generate response
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer

# Initialize the LLM
model_name = "google-t5/t5-small"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

# Prepare the input
input_text = f"Using this data: {data_str}; and, any information you believe is relevant to the user's question. Respond to this prompt: {prompt}"
inputs = tokenizer(input_text, return_tensors="pt", max_length=512, truncation=True)

# Generate the response
outputs = model.generate(**inputs, max_new_tokens=150)
response = tokenizer.decode(outputs[0], skip_special_tokens=True)
print(response)

Servo servo = Servo.get_default_servo(1) # Setting servo to 90 degrees This code configures the servo to a specific angle, which could correspond to a precise position of the attached arm.
